# Week 3 — Agent Framework

**目標**：將 Week 1 的 Skills 與 Week 2 的 RAG 整合為一個 ReAct Agent，加入多輪對話記憶。

| 步驟 | 內容 |
|------|------|
| 1 | 環境與 CUDA 確認 |
| 2 | 4-bit 模型載入 |
| 3 | 工具清單確認 |
| 4 | 建構 LangGraph ReAct Agent |
| 5 | 單問測試 |
| 6 | 多輪對話記憶測試 |
| 7 | 畫出股價圖表 |

> **架構**：文字式 ReAct（Thought→Action→Observation→Final Answer）+ LangGraph StateGraph + MemorySaver

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 1：4-bit 模型載入

In [ ]:
from src.model import load_model_4bit

tokenizer, model = load_model_4bit()
vram = torch.cuda.memory_allocated() / 1e9
print(f'\nModel loaded! VRAM used: {vram:.2f} GB')

## Step 2：確認工具清單

In [ ]:
from src.agent.agent import TOOLS

print(f'Total tools: {len(TOOLS)}\n')
for t in TOOLS:
    print(f'  [{t.name}]')
    print(f'   {t.description.split(chr(10))[0]}')
    print()

In [ ]:
# 逐一直接測試工具（不需要 LLM）
from src.tools.stock_tools import tool_get_stock_history, tool_get_fundamental_data
from src.tools.news_tools import tool_search_financial_news
from src.tools.rag_tools import tool_search_knowledge_base

print('--- Stock History (2330.TW, 1mo) ---')
print(tool_get_stock_history.invoke({'ticker': '2330.TW', 'period': '1mo'}))

print('\n--- Fundamental Data ---')
print(tool_get_fundamental_data.invoke({'ticker': '2330.TW'}))

print('\n--- Financial News ---')
print(tool_search_financial_news.invoke({'query': '2330.TW'}))

print('\n--- Knowledge Base RAG ---')
print(tool_search_knowledge_base.invoke({'query': 'ETF semiconductor'}))

## Step 3：建構 LangGraph ReAct Agent

In [ ]:
from src.agent.agent import create_agent_graph
from src.agent.memory import create_memory

memory = create_memory()  # MemorySaver — 多 thread 對話記憶
graph, memory = create_agent_graph(tokenizer, model, checkpointer=memory)

print('ReAct Agent graph created!')
print('Tools:', [t.name for t in TOOLS])

In [ ]:
# 視覺化 Agent 的節點
try:
    from IPython.display import Image
    Image(graph.get_graph().draw_mermaid_png())
except Exception as e:
    print(graph.get_graph().draw_mermaid())

## Step 4：單問測試

每個問題用 `thread_id` 隔離，`history` 初始為空。

In [ ]:
def ask(graph, question: str, thread_id: str = 'demo', history: str = '') -> dict:
    config = {'configurable': {'thread_id': thread_id}}
    init = {'input': question, 'history': history, 'scratchpad': '', 'output': '', 'iterations': 0}
    return graph.invoke(init, config=config)


print('=== Q1: TSMC 股價 ===')
r1 = ask(graph, 'What is the current price of TSMC (2330.TW)?', 'q1')
print('Scratchpad:\n', r1['scratchpad'])
print('Answer:', r1['output'])

In [ ]:
print('=== Q2: 0050 基本面 ===')
r2 = ask(graph, 'Give me the P/E ratio and dividend yield of 0050.TW', 'q2')
print('Scratchpad:\n', r2['scratchpad'])
print('Answer:', r2['output'])

In [ ]:
print('=== Q3: 台積電最新新聞 ===')
r3 = ask(graph, '台積電 (2330.TW) 最新的財經新聞有哪些？', 'q3')
print('Answer:', r3['output'])

## Step 5：多輪對話記憶測試

透過 `history` 欄位傳遞上一輪的對話記錄。

In [ ]:
print('=== Turn 1 ===')
turn1 = ask(graph, 'Help me check the stock price of 0050.TW for the past month.', 'mem-1')
print('Answer:', turn1['output'])

# Build history from turn 1
history = f"User: Help me check the stock price of 0050.TW for the past month.\nAssistant: {turn1['output']}"

print('\n=== Turn 2 (引用上輪結果) ===')
turn2 = ask(graph, 'What is the P/E ratio of that ETF you just looked up?', 'mem-1', history=history)
print('Answer:', turn2['output'])

## Step 6：畫出股價圖表

In [ ]:
r_chart = ask(graph, 'Please plot a 3-month stock chart for TSMC (2330.TW).', 'chart-1')
print(r_chart['output'])

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

charts_dir = Path('../outputs/charts')
chart_files = sorted(charts_dir.glob('*.png'), key=lambda f: f.stat().st_mtime, reverse=True)
if chart_files:
    img = mpimg.imread(chart_files[0])
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(chart_files[0].name)
    plt.show()
else:
    print('No chart found.')

## Step 7：Out-of-domain 拒絕測試（Week 5 Guardrails 預覽）

In [ ]:
r_oor = ask(graph, 'Please write me a snake game in Python.', 'oor-1')
print(r_oor['output'])

## 總結

| 任務 | 狀態 |
|------|------|
| 4-bit 模型載入（5.7 GB VRAM） | ✅ |
| 5 個工具封裝 (`@tool`) | ✅ |
| 文字式 ReAct Agent（LangGraph）| ✅ |
| MemorySaver 多輪對話記憶 | ✅ |
| 工具呼叫使用真實數據 | ✅ |
| Out-of-domain 拒絕 | ✅ |

**架構說明**
- 採用**文字式 ReAct**（非結構化 function calling），因為 8B 4-bit 量化模型對 JSON 格式工具呼叫不可靠
- `_trim_at_observation()` 防止模型在等待工具結果前自行幻覺 Observation
- `force_final` 節點確保工具結果後一定有具體答案
- 此設計限制與改善方案正是 **Week 4/5 Error Analysis** 的重點

**下一步 (Week 4)**：設計 10 題評估集（單工具、多工具、越界），量化 Tool Call 正確率、答案相關性與幻覺次數。